In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement
#%run ../delta_function ----- A decommenter pour lancer le notebook separement
#%run ./load_data ----- A decommenter pour lancer le notebook separement
#%run ./transform_data ----- A decommenter pour lancer le notebook separement

## Construction fact_batch_localization
Une ligne = une occurrence de l'evenement kiln_unload par batch. Cle primaire = batch_localization_id.

Volontairement restreinte a kiln_unload (besoin initial : recuperer la date de
fin de dechargement touraille). Pas les autres evenements (germoir, trempe...).

In [0]:
fact_batch_localization = (
    localization_events_union
    .filter(
        (F.col("batch_id").isNotNull()) &
        (F.col("localization_event") == "kiln_unload")
    )
    .select(
        F.col("site"),
        F.col("prd_line"),
        F.col("prd_workshop"),
        F.col("prd_cell"),
        F.col("batch_id"),
        F.col("localization_event"),
        F.col("start"),
        F.col("end")
    )
    .withColumn("end_date", F.to_date(F.col("end")))
    .withColumn(
        "batch_localization_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("site"),
                F.col("batch_id"),
                F.col("localization_event"),
                F.col("prd_cell").cast("string"),
                F.coalesce(F.col("start").cast("string"), F.lit("")),
                F.coalesce(F.col("end").cast("string"), F.lit(""))
            ),
            256
        )
    )
)

# Cle technique generee par hash (pas de cle unique native cote source) :
# deterministe donc stable d'une execution a l'autre, ce qui permet un
# merge/upsert correct au lieu de dupliquer les lignes a chaque run.

## Derniere localisation par batch/production_line/localization_event
Garde une seule ligne par (batch_id, prd_line, localization_event) : celle avec la end date la plus recente.
Utile pour savoir dans quelle cellule/germoir se trouve actuellement (ou en dernier) un batch, pour chaque type d'evenement.

In [0]:
window_last_localization = Window.partitionBy("batch_id", "prd_line", "localization_event").orderBy(F.col("end").desc())

fact_batch_localization = (
    fact_batch_localization
    .withColumn("rn", F.row_number().over(window_last_localization))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

## Ecriture Delta

In [0]:
current_process = "fact_batch_localization"
target_fact_batch_localization = current_catalog + "." + current_schema + "." + current_process
print(target_fact_batch_localization)

In [0]:
all_columns = fact_batch_localization.columns
primary_key = ['batch_localization_id']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
handle_table_update(
    fact_batch_localization,
    target_fact_batch_localization,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)